In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

# --------------------------
# Dataset
# --------------------------
class MNISTCSVDataset(Dataset):
    def __init__(self, csv_file):
        data = pd.read_csv(csv_file)
        self.labels = data.iloc[:, 0].values.astype(np.int64)
        self.images = data.iloc[:, 1:].values.astype(np.float32) / 255.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx].reshape(28, 28)
        label = self.labels[idx]
        image = torch.tensor(image).float().unsqueeze(0)
        label = torch.tensor(label).long()
        return image, label

# --------------------------
# Model
# --------------------------
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 4),
            nn.ReLU(),
            nn.Linear(4, 4),
            nn.ReLU(),
            nn.Linear(4, 10)
        )

    def forward(self, x):
        return self.network(x)

# --------------------------
# Training function
# --------------------------
def train_model():
    train_dataset = MNISTCSVDataset("mnist_train.csv")
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    model = MNISTModel()
    model = model.float()  # Ensure FP32

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    epochs = 50
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} Loss {total_loss:.4f}")

    return model

# --------------------------
# Export to ONNX
# --------------------------
def export_onnx(model):
    os.makedirs("models", exist_ok=True)
    dummy_input = torch.randn(1, 1, 28, 28)
    torch.onnx.export(
        model,
        dummy_input,
        "models/mnist_model.onnx",
        input_names=["input"],
        output_names=["output"],
        opset_version=11
    )
    print("ONNX model exported to models/mnist_model.onnx")

# --------------------------
# Main
# --------------------------
if __name__ == "__main__":
    model = train_model()
    export_onnx(model)

Epoch 0 Loss 171.8020
Epoch 1 Loss 114.4462
Epoch 2 Loss 87.4179
Epoch 3 Loss 72.2577
Epoch 4 Loss 65.5542
Epoch 5 Loss 60.8115
Epoch 6 Loss 58.3638
Epoch 7 Loss 55.4471
Epoch 8 Loss 53.9460
Epoch 9 Loss 52.4916
Epoch 10 Loss 51.8588
Epoch 11 Loss 50.1813
Epoch 12 Loss 49.8243
Epoch 13 Loss 49.0841
Epoch 14 Loss 48.5964
Epoch 15 Loss 47.2184
Epoch 16 Loss 47.0238
Epoch 17 Loss 46.5034
Epoch 18 Loss 45.4184
Epoch 19 Loss 45.0983
Epoch 20 Loss 45.8541
Epoch 21 Loss 44.1733
Epoch 22 Loss 45.2018
Epoch 23 Loss 44.6289
Epoch 24 Loss 44.0437
Epoch 25 Loss 43.5834
Epoch 26 Loss 43.2685
Epoch 27 Loss 41.9643
Epoch 28 Loss 42.5881
Epoch 29 Loss 41.7812
Epoch 30 Loss 41.4701
Epoch 31 Loss 41.5172
Epoch 32 Loss 41.1102
Epoch 33 Loss 40.0736
Epoch 34 Loss 40.3452
Epoch 35 Loss 40.3504
Epoch 36 Loss 40.0011
Epoch 37 Loss 40.1503
Epoch 38 Loss 39.8053
Epoch 39 Loss 39.0118
Epoch 40 Loss 40.6385
Epoch 41 Loss 39.0623
Epoch 42 Loss 38.3841
Epoch 43 Loss 38.1248
Epoch 44 Loss 38.9432
Epoch 45 Loss 40.1

In [2]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxconverter_common import float16
import os

os.makedirs("models", exist_ok=True)

fp32_model = "models/mnist_model.onnx"

# --------------------------
# FP16 Quantization
# --------------------------
model = onnx.load(fp32_model)
fp16_model = float16.convert_float_to_float16(model)
onnx.save(fp16_model, "models/mnist_model_quantized_fp16.onnx")
print("FP16 model saved.")

# --------------------------
# INT8 Quantization
# --------------------------
quantize_dynamic(
    fp32_model,
    "models/mnist_model_quantized_int8.onnx",
    weight_type=QuantType.QInt8
)
print("INT8 model saved.")

FP16 model saved.
INT8 model saved.


In [3]:
import onnxruntime as ort
import pandas as pd
import numpy as np
import glob
import os


# --------------------------
# Load MNIST Test Data
# --------------------------
def load_dataset():
    data = pd.read_csv("mnist_test.csv")
    labels = data.iloc[:, 0].values
    images = data.iloc[:, 1:].values.astype(np.float32) / 255.0
    images = images.reshape(-1, 1, 28, 28)
    return images, labels


images, labels = load_dataset()

# --------------------------
# Evaluate all ONNX models in models/ folder
# --------------------------
onnx_files = glob.glob("models/*.onnx")

for model_file in onnx_files:
    session = ort.InferenceSession(model_file)
    input_name = session.get_inputs()[0].name
    input_type = session.get_inputs()[0].type

    correct = 0

    for i in range(len(images)):
        img = images[i : i + 1]

        if "float16" in input_type:
            img = img.astype(np.float16)

        outputs = session.run(None, {input_name: img})
        pred = np.argmax(outputs[0])
        if pred == labels[i]:
            correct += 1

    accuracy = correct / len(images)
    print(f"{os.path.basename(model_file)} Accuracy: {accuracy:.4f}")

mnist_model_quantized_fp16.onnx Accuracy: 0.7650
mnist_model_quantized_int8.onnx Accuracy: 0.7640
mnist_model.onnx Accuracy: 0.7660


In [4]:
import onnx

# 检查模型信息
def check_model_info(model_path):
    model = onnx.load(model_path)
    print(f"\nModel: {model_path}")
    print(f"Number of nodes: {len(model.graph.node)}")

    # 检查权重类型
    for tensor in model.graph.initializer:
        if tensor.data_type == 1:  # FLOAT
            print(f"  Weight: {tensor.name} - FLOAT")
        elif tensor.data_type == 10:  # FLOAT16
            print(f"  Weight: {tensor.name} - FLOAT16")
        elif tensor.data_type == 3:  # INT8
            print(f"  Weight: {tensor.name} - INT8")


# 检查所有模型
for model_file in onnx_files:
    check_model_info(model_file)


Model: models/mnist_model_quantized_fp16.onnx
Number of nodes: 6
  Weight: network.1.weight - FLOAT16
  Weight: network.1.bias - FLOAT16
  Weight: network.3.weight - FLOAT16
  Weight: network.3.bias - FLOAT16
  Weight: network.5.weight - FLOAT16
  Weight: network.5.bias - FLOAT16

Model: models/mnist_model_quantized_int8.onnx
Number of nodes: 21
  Weight: network.1.bias - FLOAT
  Weight: network.3.bias - FLOAT
  Weight: network.5.bias - FLOAT
  Weight: network.1.weight_scale - FLOAT
  Weight: network.1.weight_zero_point - INT8
  Weight: network.1.weight_quantized - INT8
  Weight: network.3.weight_scale - FLOAT
  Weight: network.3.weight_zero_point - INT8
  Weight: network.3.weight_quantized - INT8
  Weight: network.5.weight_scale - FLOAT
  Weight: network.5.weight_zero_point - INT8
  Weight: network.5.weight_quantized - INT8

Model: models/mnist_model.onnx
Number of nodes: 6
  Weight: network.1.weight - FLOAT
  Weight: network.1.bias - FLOAT
  Weight: network.3.weight - FLOAT
  Weight

In [5]:
def compare_outputs(model1_path, model2_path, num_samples=10):
    # 加载两个模型
    session1 = ort.InferenceSession(model1_path)
    session2 = ort.InferenceSession(model2_path)

    input_name1 = session1.get_inputs()[0].name
    input_name2 = session2.get_inputs()[0].name

    # 准备相同输入
    np.random.seed(42)
    dummy_input = np.random.randn(num_samples, 1, 28, 28).astype(np.float32)

    # 转换数据类型
    if "float16" in session1.get_inputs()[0].type:
        input1 = dummy_input.astype(np.float16)
    else:
        input1 = dummy_input.astype(np.float32)

    if "float16" in session2.get_inputs()[0].type:
        input2 = dummy_input.astype(np.float16)
    else:
        input2 = dummy_input.astype(np.float32)

    # 获取输出
    outputs1 = []
    outputs2 = []

    for i in range(num_samples):
        out1 = session1.run(None, {input_name1: input1[i : i + 1]})
        out2 = session2.run(None, {input_name2: input2[i : i + 1]})
        outputs1.append(out1[0])
        outputs2.append(out2[0])

    # 比较输出差异
    total_diff = 0
    for i in range(num_samples):
        diff = np.abs(outputs1[i] - outputs2[i]).mean()
        total_diff += diff

    avg_diff = total_diff / num_samples
    return avg_diff


# 比较原始模型与量化模型
print(
    "原始 vs FP16:",
    compare_outputs(
        "models/mnist_model.onnx", "models/mnist_model_quantized_fp16.onnx"
    ),
)
print(
    "原始 vs INT8:",
    compare_outputs(
        "models/mnist_model.onnx", "models/mnist_model_quantized_int8.onnx"
    ),
)

原始 vs FP16: 0.032303186738863586
原始 vs INT8: 0.9632193833589554
